# Hypothesis: Pre-Draft Background and Early NBA Performance

## Research Question

Does a player's pre-draft background influence his performance during the first three seasons in the NBA?

## Hypothesis

Players with stronger pre-draft characteristics tend to perform better during their first three NBA seasons.

In this project, pre-draft background includes draft position, draft round, physical measurements, and available combine statistics. Early NBA performance is measured using average player statistics during the first three seasons after entering the league.


## Key Variables

### Pre-Draft Characteristics

Possible explanatory variables:

* draft number
* height
* weight
* wingspan
* standing reach
* vertical leap
* position
* college or previous team

### Early NBA Performance

Performance during the first three NBA seasons can be measured using:

* average points per game
* average assists per game
* average rebounds per game
* average minutes played
* average games played in the season




In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [21]:
draft_history_path = "..\data\processed\draft_history.csv"
draft_history = pd.read_csv(draft_history_path)
combine_draft_history_path = "..\data\processed\draft_combine_stats.csv"
combine_draft_history = pd.read_csv(combine_draft_history_path)

### Players

In [24]:
players = draft_history[draft_history["round_number"] == 1][["person_id", "player_name", "season", "overall_pick", "organization", "organization_type"]]
сombine_draft = combine_draft_history[['player_id','height_wo_shoes', 'weight', 'wingspan']]

In [27]:
players = players.rename(columns={'person_id': 'player_id'})
players = players.merge(
    сombine_draft,
    on="player_id",
    how="left"
)

In [37]:
players["weight"] = pd.to_numeric(players["weight"], errors="coerce")
players["imb"] = players["weight"] * 703 / players["height_wo_shoes"]**2


In [39]:
players

,player_id,player_name,season,overall_pick,organization,organization_type,height_wo_shoes,weight,wingspan,imb
0,79299,Clifton McNeeley,1947,1,Texas-El Paso,College/University,NaN,NaN,NaN,NaN
1,78109,Glen Selbo,1947,2,Wisconsin,College/University,NaN,NaN,NaN,NaN
2,76649,Eddie Ehlers,1947,3,Purdue,College/University,NaN,NaN,NaN,NaN
3,79302,Walt Dropo,1947,4,Connecticut,College/University,NaN,NaN,NaN,NaN
4,77048,Dick Holub,1947,5,Long Island-Brooklyn,College/University,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1629,1641733,Nick Smith Jr.,2023,27,Arkansas,College/University,77.75,NaN,82.50,NaN
1630,1641729,Brice Sensabaugh,2023,28,Ohio State,College/University,77.75,NaN,82.50,NaN
1631,1631124,Julian Strawther,2023,29,Gonzaga,College/University,77.75,207.6,81.50,24.142480
1632,1631124,Julian Strawther,2023,29,Gonzaga,College/University,78.00,208.8,81.25,24.126627


### Work with game stats of players

In [100]:
pbp_path = "..\data\processed\play_by_play.csv"
pbp = pd.read_csv(pbp_path)

In [101]:
field_goals = pbp[pbp["eventmsgtype"] == 1]
free_throws = pbp[pbp["eventmsgtype"] == 3]
rebounds = pbp[pbp["eventmsgtype"] == 4]

In [104]:
field_goals["description"] = (
    field_goals["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(field_goals["visitordescription"].replace("unknown", pd.NA))
)

free_throws["description"] = (
    free_throws["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(free_throws["visitordescription"].replace("unknown", pd.NA))
)

rebounds["description"] = (
    rebounds["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(rebounds["visitordescription"].replace("unknown", pd.NA))
)

In [ ]:
field_goals["pts"] = 2
field_goals.loc[
    field_goals["description"].str.contains("3PT"),
    "pts"
] = 3

free_throws = free_throws[~free_throws['description'].str.contains('MISS')].copy()
free_throws["pts"] = 1



In [121]:
scoring_events = pd.concat([
    field_goals[["game_id", "player1_id", "player2_id", "pts"]],
    free_throws[["game_id", "player1_id", "pts"]]
], ignore_index=True).sort_values(by='game_id')

rebounds = rebounds[["game_id", 'player1_id']]


In [119]:
scoring_events



,game_id,player1_id,player2_id,pts
2938809,11300001,42545,NaN,1
2938810,11300001,200757,NaN,1
2938811,11300001,42544,NaN,1
2938812,11300001,42544,NaN,1
2938813,11300001,42544,NaN,1
...,...,...,...,...
2352559,49800087,764,NaN,1
2352556,49800087,990,NaN,1
2352557,49800087,1495,NaN,1
2352536,49800087,251,NaN,1


In [122]:
rebounds

,game_id,player1_id
3,29600012,406
5,29600012,208
9,29600012,1610612747
14,29600012,76
17,29600012,170
...,...,...
13585524,32200001,202681
13585527,32200001,1610616834
13585529,32200001,1628374
13585531,32200001,203944
